In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, current_timestamp

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [4]:
spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark com Iceberg iniciado!")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-840feba3-d7b9-4c2a-9d45-2bd55d28d719;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.6.0/postgresql-42.6.0.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.6.0!postgresql.jar (150ms)
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.4.2/iceberg-spark-runtime-3.5_2.12-1.4.2.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2!iceberg-spark-runtime-3.5_2.12.jar (583ms)
downloading https://repo1.ma

✅ Spark com Iceberg iniciado!


In [5]:
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

DataFrame[]

In [ ]:
df_raw = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/pipeline_transacoes") \
    .option("dbtable", "staging.transacoes_raw") \
    .option("user", "postgres") \
    .option("password", "1234") \
    .option("driver", "org.postgresql.Driver") \
    .load()
spark.sparkContext.setLogLevel("ERROR")
df_raw.show(5)

+---+--------------------+--------------------+
| id|             payload|       data_ingestao|
+---+--------------------+--------------------+
|  1|{"valor": 2321.91...|2026-03-29 23:09:...|
|  2|{"valor": 4130.44...|2026-03-29 23:09:...|
|  3|{"valor": 4633.68...|2026-03-29 23:09:...|
|  4|{"valor": 3685.2,...|2026-03-29 23:09:...|
|  5|{"valor": 1817.05...|2026-03-29 23:09:...|
+---+--------------------+--------------------+
only showing top 5 rows



In [7]:
df_bronze = df_raw.withColumn(
    "data_ingestao",
    current_timestamp()
).withColumn(
    "data_particao",
    to_date(col("data_ingestao"))
)

df_bronze.show(5)

+---+--------------------+--------------------+-------------+
| id|             payload|       data_ingestao|data_particao|
+---+--------------------+--------------------+-------------+
|  1|{"valor": 2321.91...|2026-03-29 23:31:...|   2026-03-29|
|  2|{"valor": 4130.44...|2026-03-29 23:31:...|   2026-03-29|
|  3|{"valor": 4633.68...|2026-03-29 23:31:...|   2026-03-29|
|  4|{"valor": 3685.2,...|2026-03-29 23:31:...|   2026-03-29|
|  5|{"valor": 1817.05...|2026-03-29 23:31:...|   2026-03-29|
+---+--------------------+--------------------+-------------+
only showing top 5 rows



In [8]:
df_bronze.writeTo("lakehouse.bronze.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()

print("✅ Dados gravados na Bronze")

✅ Dados gravados na Bronze


In [9]:
df_bronze.writeTo("lakehouse.bronze.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()

print("✅ Dados gravados na Bronze")

✅ Dados gravados na Bronze


In [10]:
spark.sql("SELECT * FROM lakehouse.bronze.transacoes.partitions").show()

+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+--------------------+------------------------+
|   partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|     last_updated_at|last_updated_snapshot_id|
+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+--------------------+------------------------+
|{2026-03-29}|      0|          93|         1|                         6267|                           0|                         0|                           0|                         0|2026-03-29 23:31:...|     1677005071362373099|
+------------+-------+------------+----------+--------------

In [11]:
!ls /tmp/warehouse/bronze/transacoes

data  metadata
